# Adaptive Evidence Graph Learning (AEGL) — Completed Reference Notebook

This is the cleaned, consolidated, and completed version of the original AEGL
prototype notebook. The original had three incompatible copies of the same
classes scattered across cells, a syntax error in the first cell, no
aleatoric/epistemic uncertainty split despite the proposal promising one, no
calibration metric, and only ever trained on random noise tensors instead of
real images.

This notebook fixes all of that and is a **companion reference** to the full
app in `aegl/backend/` + `aegl/frontend/` — same model code, same fixes,
just runnable interactively here too, with real long-tail digit images
instead of `torch.randn(...)` placeholders.


In [1]:
"""
Adaptive Evidence Graph Learning (AEGL) — Core Engine
=======================================================
This module is the completed, consolidated implementation of the architecture
described in the AEGL concept note. It reconciles the multiple draft variants
found in the original notebook into a single, correct, runnable system and
fills in the pieces the proposal called for but that were never implemented:

  * A real (small) vision encoder over image tensors (not just flat vectors).
  * A differentiable structural router: soft compatibility scores -> Gumbel-Softmax
    relaxation during training, sparse top-k masked-attention routing at inference.
  * A Dynamic Graph Transformer that performs masked message passing strictly
    over the routed topology (the earlier "multiply-after-softmax" gating leaked
    attention mass to unrouted nodes — fixed here with a pre-softmax mask, matching
    the "FIX" already sketched in the last notebook cell).
  * An explicit Evidence Lower Bound (ELBO) loss: L = CE + beta * KL(q(G|x,M) || p(G)).
  * A proper epistemic/aleatoric uncertainty DECOMPOSITION via MC-Dropout (BALD):
        total (predictive) entropy = H[ E_t[p(y|x,G_t)] ]
        aleatoric                  = E_t[ H[p(y|x,G_t)] ]
        epistemic (mutual info)    = total - aleatoric
    The original notebook only ever computed a single ad-hoc "epistemic" number
    (predictive entropy + graph-entropy variance) and never separated aleatoric
    uncertainty at all, despite the proposal explicitly promising both.
  * Expected Calibration Error (ECE) — the proposal's stated success metric
    ("reducing expected calibration error") was never implemented anywhere.
  * A Causal Counterfactual Verifier that measures how much masking the top
    routed evidence edges deflects the prediction (causal attribution score).
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================================
# 1. VISION ENCODER
# ============================================================================
class VisionEncoder(nn.Module):
    """Small convolutional encoder producing a latent feature token z_x.

    Standing in for a frozen foundation backbone (CLIP / DINOv2) as described
    in the proposal — swap this module out for a real frozen backbone without
    touching anything downstream.
    """

    def __init__(self, in_channels: int, feature_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64, feature_dim),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================================
# 2. DYNAMIC GRAPH TRANSFORMER (masked message passing over routed topology)
# ============================================================================
class DynamicGraphTransformerLayer(nn.Module):
    def __init__(self, feature_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads

        self.q_proj = nn.Linear(feature_dim, feature_dim, bias=False)
        self.k_proj = nn.Linear(feature_dim, feature_dim, bias=False)
        self.v_proj = nn.Linear(feature_dim, feature_dim, bias=False)
        self.out_proj = nn.Linear(feature_dim, feature_dim, bias=False)

        self.ffn = nn.Sequential(
            nn.Linear(feature_dim, feature_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 2, feature_dim),
        )

        self.norm1 = nn.LayerNorm(feature_dim)
        self.norm2 = nn.LayerNorm(feature_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, nodes, adjacency_matrix):
        """
        nodes: [B, N, D]   adjacency_matrix: [B, N, N] (soft, in [0,1], 0 = no edge)
        """
        batch_size, num_nodes, _ = nodes.size()
        residual = nodes

        q = self.q_proj(nodes).view(batch_size, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(nodes).view(batch_size, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(nodes).view(batch_size, num_nodes, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Structural mask: suppress (not merely down-weight) attention to unrouted edges,
        # applied BEFORE softmax so probability mass cannot leak to non-routed nodes.
        gated_adjacency = adjacency_matrix.unsqueeze(1)  # [B, 1, N, N]
        structural_mask = (gated_adjacency <= 1e-8)
        attn_scores = attn_scores.masked_fill(structural_mask, float("-1e9"))
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        context = torch.matmul(attn_probs, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, num_nodes, self.feature_dim)

        x = self.norm1(residual + self.out_proj(context))
        ffn_out = self.ffn(x)
        refined_nodes = self.norm2(x + self.dropout(ffn_out))
        return refined_nodes


# ============================================================================
# 3. THE UNIFIED AEGL SYSTEM (encoder -> differentiable router -> graph transformer -> head)
# ============================================================================
class AEGLSystem(nn.Module):
    def __init__(self, in_channels, feature_dim, memory_size, num_classes,
                 tau=1.0, top_k=8, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.memory_size = memory_size
        self.tau = tau
        self.top_k = min(top_k, memory_size)

        self.vision_encoder = VisionEncoder(in_channels, feature_dim, dropout=dropout)
        self.memory_repository = nn.Parameter(torch.randn(memory_size, feature_dim) * 0.5)
        self.graph_transformer = DynamicGraphTransformerLayer(feature_dim, dropout=dropout)
        self.classification_head = nn.Linear(feature_dim, num_classes)

    def differentiable_router(self, z_x):
        """q(G | x, M): soft compatibility -> Gumbel-Softmax relaxation (train) /
        sparse top-k masked routing (eval)."""
        scores = torch.matmul(z_x, self.memory_repository.t()) / (self.feature_dim ** 0.5)
        omega = F.softmax(scores, dim=-1)

        if self.training:
            gumbel_noise = -torch.log(-torch.log(torch.rand_like(scores) + 1e-20) + 1e-20)
            logits = (torch.log(omega + 1e-20) + gumbel_noise) / self.tau
            a_route = F.softmax(logits, dim=-1)
        else:
            topk_vals, topk_idx = torch.topk(omega, self.top_k, dim=-1)
            sparse_mask = torch.zeros_like(omega).scatter_(-1, topk_idx, 1.0)
            a_route = F.softmax(scores.masked_fill(sparse_mask == 0, float("-1e9")), dim=-1)

        return a_route

    def forward(self, x, return_nodes=False):
        batch_size = x.size(0)
        z_x = self.vision_encoder(x)
        a_route = self.differentiable_router(z_x)

        image_token = z_x.unsqueeze(1)
        expanded_memory = self.memory_repository.unsqueeze(0).expand(batch_size, -1, -1)
        combined_nodes = torch.cat([image_token, expanded_memory], dim=1)

        full_adj = torch.zeros(batch_size, 1 + self.memory_size, 1 + self.memory_size, device=x.device)
        full_adj[:, 0, 1:] = a_route
        full_adj[:, 1:, 0] = a_route
        # every node needs a self-loop or fully-masked rows produce NaNs in softmax
        idx = torch.arange(1 + self.memory_size, device=x.device)
        full_adj[:, idx, idx] = 1.0

        refined_nodes = self.graph_transformer(combined_nodes, full_adj)
        final_image_state = refined_nodes[:, 0, :]
        logits = self.classification_head(final_image_state)

        if return_nodes:
            return logits, a_route, refined_nodes
        return logits, a_route


# ============================================================================
# 4. VARIATIONAL ELBO LOSS ENGINE
# ============================================================================
class AEGLLossEngine(nn.Module):
    """L_ELBO = CrossEntropy(y, y_hat) + beta * D_KL( q(G|x,M) || p(G) )"""

    def __init__(self, beta=1.0, epsilon=1e-10):
        super().__init__()
        self.beta = beta
        self.epsilon = epsilon

    def compute_kl_divergence(self, q_adjacency, p_prior=None):
        if p_prior is None:
            num_nodes = q_adjacency.size(-1)
            p_prior = torch.full_like(q_adjacency, 1.0 / num_nodes)
        kl_val = q_adjacency * (torch.log(q_adjacency + self.epsilon) - torch.log(p_prior + self.epsilon))
        return torch.mean(torch.sum(kl_val, dim=-1))

    def forward(self, logits, targets, adjacency_matrix, p_prior=None):
        ce_loss = F.cross_entropy(logits, targets)
        kl_loss = self.compute_kl_divergence(adjacency_matrix, p_prior)
        total_loss = ce_loss + self.beta * kl_loss
        return total_loss, ce_loss, kl_loss


# ============================================================================
# 5. UNCERTAINTY ENGINE — proper epistemic / aleatoric decomposition (BALD)
# ============================================================================
class UncertaintyEngine:
    """MC-Dropout based decomposition of predictive uncertainty.

    total (predictive) entropy = H[ mean_t p_t ]
    aleatoric                  = mean_t H[ p_t ]
    epistemic (mutual info)    = total - aleatoric
    """

    def __init__(self, num_samples: int = 12, epsilon: float = 1e-10):
        self.num_samples = num_samples
        self.epsilon = epsilon

    @staticmethod
    def _entropy(probs, eps):
        return -torch.sum(probs * torch.log(probs + eps), dim=-1)

    @torch.no_grad()
    def estimate(self, model: AEGLSystem, x: torch.Tensor):
        was_training = model.training
        model.train()  # enable dropout stochasticity for MC sampling
        # keep BatchNorm-free architecture so train() only toggles dropout masks

        prob_samples = []
        adj_samples = []
        for _ in range(self.num_samples):
            logits, adj = model(x)
            prob_samples.append(F.softmax(logits, dim=-1))
            adj_samples.append(adj)

        model.train(was_training)

        probs_stack = torch.stack(prob_samples, dim=0)      # [T, B, C]
        mean_probs = probs_stack.mean(dim=0)                # [B, C]

        total_entropy = self._entropy(mean_probs, self.epsilon)                     # [B]
        per_sample_entropy = self._entropy(probs_stack, self.epsilon)               # [T, B]
        aleatoric = per_sample_entropy.mean(dim=0)                                  # [B]
        epistemic = (total_entropy - aleatoric).clamp(min=0.0)                       # [B]

        mean_adjacency = torch.stack(adj_samples, dim=0).mean(dim=0)  # [B, memory_size]

        return {
            "mean_probs": mean_probs,
            "total_uncertainty": total_entropy,
            "aleatoric_uncertainty": aleatoric,
            "epistemic_uncertainty": epistemic,
            "mean_adjacency": mean_adjacency,
        }


def expected_calibration_error(probs: torch.Tensor, targets: torch.Tensor, n_bins: int = 10):
    """Standard binned ECE — the calibration metric the proposal names as a
    target benchmark but never implements."""
    confidences, predictions = torch.max(probs, dim=-1)
    accuracies = predictions.eq(targets)

    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    ece = torch.zeros(1, device=probs.device)
    bin_stats = []

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi)
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            acc_in_bin = accuracies[in_bin].float().mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += torch.abs(conf_in_bin - acc_in_bin) * prop_in_bin
            bin_stats.append({
                "bin_lower": lo.item(), "bin_upper": hi.item(),
                "accuracy": acc_in_bin.item(), "confidence": conf_in_bin.item(),
                "proportion": prop_in_bin.item(),
            })
        else:
            bin_stats.append({
                "bin_lower": lo.item(), "bin_upper": hi.item(),
                "accuracy": 0.0, "confidence": 0.0, "proportion": 0.0,
            })

    return ece.item(), bin_stats


# ============================================================================
# 6. CAUSAL COUNTERFACTUAL VERIFIER
# ============================================================================
class AEGLCausalVerifier(nn.Module):
    """Stress-tests the routed subgraph: masks out the top evidence edges and
    measures how far predictions deflect (KL between original and counterfactual
    predictive distributions). A high score means the routed topology was
    causally load-bearing for the decision — the "auditable structural
    reasoning" the proposal calls for."""

    def __init__(self, alpha=0.15):
        super().__init__()
        self.alpha = alpha

    def generate_counterfactual_topology(self, adjacency_matrix, perturbation_type="edge_mask"):
        counterfactual_adj = adjacency_matrix.clone()
        if perturbation_type == "edge_mask":
            threshold = torch.quantile(adjacency_matrix, 1.0 - self.alpha)
            mask = adjacency_matrix < threshold
            counterfactual_adj = counterfactual_adj * mask.float()
        elif perturbation_type == "gaussian_noise":
            noise = torch.randn_like(adjacency_matrix) * self.alpha
            counterfactual_adj = torch.clamp(adjacency_matrix + noise, min=0.0, max=1.0)
        return counterfactual_adj

    @torch.no_grad()
    def forward(self, aegl_system: AEGLSystem, inputs, base_logits, base_adjacency):
        was_training = aegl_system.training
        aegl_system.eval()

        cf_adjacency = self.generate_counterfactual_topology(base_adjacency, "edge_mask")

        z_x = aegl_system.vision_encoder(inputs)
        batch_size = inputs.size(0)
        image_token = z_x.unsqueeze(1)
        expanded_memory = aegl_system.memory_repository.unsqueeze(0).expand(batch_size, -1, -1)
        combined_nodes = torch.cat([image_token, expanded_memory], dim=1)

        cf_full_adj = torch.zeros(batch_size, 1 + aegl_system.memory_size,
                                   1 + aegl_system.memory_size, device=inputs.device)
        cf_full_adj[:, 0, 1:] = cf_adjacency
        cf_full_adj[:, 1:, 0] = cf_adjacency
        idx = torch.arange(1 + aegl_system.memory_size, device=inputs.device)
        cf_full_adj[:, idx, idx] = 1.0

        cf_refined_nodes = aegl_system.graph_transformer(combined_nodes, cf_full_adj)
        cf_logits = aegl_system.classification_head(cf_refined_nodes[:, 0, :])

        base_probs = F.softmax(base_logits, dim=-1)
        cf_probs = F.softmax(cf_logits, dim=-1)
        causal_attribution_score = torch.sum(
            base_probs * (torch.log(base_probs + 1e-10) - torch.log(cf_probs + 1e-10)), dim=-1
        )

        aegl_system.train(was_training)
        return causal_attribution_score


## Real long-tail dataset

Instead of the original's `torch.randn(...)` dummy batches, we use
scikit-learn's `digits` dataset (real 8×8 handwritten digits) resampled into
an artificial long-tail class distribution — an open-world benchmark stand-in
that needs no network download.


In [2]:
"""
Data engine for AEGL demo runs.

Uses scikit-learn's `digits` dataset (real 8x8 handwritten digit images, no
network download required) and reshapes it into an open-world / long-tail
benchmark by sub-sampling some classes far more aggressively than others —
mirroring the ImageNet-A/R / iNaturalist heavy-tail scenario named in the
proposal, without needing an external dataset download (this sandbox's
network egress is restricted to package registries only).

Also supports user-uploaded CSV files of flattened pixel rows + a label
column, so the frontend's "upload your training file" panel is real, not
just decorative.
"""

import io
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.datasets import load_digits


class LongTailDigitsDataset(Dataset):
    """Real 8x8 digit images (0-9), artificially long-tailed by class."""

    def __init__(self, split="train", imbalance_ratio=8.0, test_size=0.25, seed=42):
        data = load_digits()
        X, y = data.images.astype(np.float32), data.target.astype(np.int64)

        rng = np.random.RandomState(seed)
        perm = rng.permutation(len(X))
        X, y = X[perm], y[perm]

        n_test = int(len(X) * test_size)
        if split == "test":
            X, y = X[:n_test], y[:n_test]
        else:
            X, y = X[n_test:], y[n_test:]

        if split == "train":
            X, y = self._apply_long_tail(X, y, imbalance_ratio, rng)

        # normalize to [0, 1], add channel dim -> [N, 1, 8, 8]
        X = X / 16.0
        self.images = torch.from_numpy(X).unsqueeze(1)
        self.labels = torch.from_numpy(y)
        self.num_classes = int(y.max()) + 1 if len(y) else 10
        self.class_counts = np.bincount(y, minlength=10).tolist()

    @staticmethod
    def _apply_long_tail(X, y, imbalance_ratio, rng):
        classes = np.unique(y)
        n_classes = len(classes)
        max_count = max(np.sum(y == c) for c in classes)
        keep_idx = []
        for i, c in enumerate(classes):
            # exponential decay in per-class sample count -> heavy tail
            frac = imbalance_ratio ** (-i / max(1, n_classes - 1))
            target_count = max(3, int(max_count * frac))
            cls_idx = np.where(y == c)[0]
            rng.shuffle(cls_idx)
            keep_idx.extend(cls_idx[:target_count].tolist())
        keep_idx = np.array(keep_idx)
        rng.shuffle(keep_idx)
        return X[keep_idx], y[keep_idx]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


class UploadedCSVDataset(Dataset):
    """CSV with one label column ('label' or first column) and the rest as
    flattened pixel intensities. Values are auto-reshaped to a square image
    where possible, otherwise treated as a 1D feature vector padded to a
    small square for the conv encoder."""

    def __init__(self, csv_bytes: bytes):
        import csv as _csv
        text = csv_bytes.decode("utf-8", errors="ignore")
        reader = list(_csv.reader(io.StringIO(text)))
        header = reader[0]
        rows = reader[1:]

        label_col = 0
        for i, h in enumerate(header):
            if h.strip().lower() in ("label", "target", "class", "y"):
                label_col = i
                break

        labels = []
        features = []
        for row in rows:
            if not row:
                continue
            labels.append(int(float(row[label_col])))
            feat = [float(v) for j, v in enumerate(row) if j != label_col]
            features.append(feat)

        X = np.array(features, dtype=np.float32)
        y = np.array(labels, dtype=np.int64)

        # min-max normalize
        if X.max() > X.min():
            X = (X - X.min()) / (X.max() - X.min())

        side = int(np.ceil(np.sqrt(X.shape[1])))
        padded = np.zeros((X.shape[0], side * side), dtype=np.float32)
        padded[:, :X.shape[1]] = X
        X = padded.reshape(-1, 1, side, side)

        # remap labels to a dense 0..K-1 range
        unique_labels = sorted(set(y.tolist()))
        remap = {lbl: i for i, lbl in enumerate(unique_labels)}
        y = np.array([remap[v] for v in y.tolist()], dtype=np.int64)

        self.images = torch.from_numpy(X)
        self.labels = torch.from_numpy(y)
        self.num_classes = len(unique_labels)
        self.class_counts = np.bincount(y, minlength=self.num_classes).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


## End-to-end training + evaluation run

Trains `AEGLSystem` with the completed ELBO loss, then runs:
- held-out accuracy,
- Expected Calibration Error (missing from the original, now implemented),
- the proper epistemic/aleatoric uncertainty decomposition via MC-Dropout,
- causal counterfactual verification (edge-masking stress test).


In [3]:

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_set = LongTailDigitsDataset(split="train", imbalance_ratio=8.0)
test_set = LongTailDigitsDataset(split="test", imbalance_ratio=8.0)
print(f"Train size: {len(train_set)} | class counts: {train_set.class_counts}")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

model = AEGLSystem(in_channels=1, feature_dim=64, memory_size=256,
                    num_classes=train_set.num_classes, tau=1.0, top_k=8).to(device)
criterion = AEGLLossEngine(beta=1.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
verifier = AEGLCausalVerifier(alpha=0.15)
unc_engine = UncertaintyEngine(num_samples=10)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = ep_ce = ep_kl = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, adj = model(images)
        loss, ce, kl = criterion(logits, targets, adj)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        ep_loss += loss.item(); ep_ce += ce.item(); ep_kl += kl.item()
    n = len(train_loader)
    print(f"Epoch [{epoch}/{EPOCHS}] ELBO: {ep_loss/n:.4f} | CE: {ep_ce/n:.4f} | KL: {ep_kl/n:.4f}")


Device: cpu
Train size: 619 | class counts: [135, 115, 91, 72, 57, 45, 36, 28, 22, 18]


Epoch [1/10] ELBO: 4.2427 | CE: 2.1984 | KL: 2.0443


Epoch [2/10] ELBO: 4.1829 | CE: 2.1148 | KL: 2.0681


Epoch [3/10] ELBO: 4.0655 | CE: 2.0479 | KL: 2.0176


Epoch [4/10] ELBO: 3.6366 | CE: 1.6134 | KL: 2.0231


...

Epoch [6/10] ELBO: 2.9766 | CE: 0.9528 | KL: 2.0238


Epoch [7/10] ELBO: 2.8813 | CE: 0.8241 | KL: 2.0572


Epoch [8/10] ELBO: 2.7096 | CE: 0.6244 | KL: 2.0852


Epoch [9/10] ELBO: 2.5868 | CE: 0.5454 | KL: 2.0413


Epoch [10/10] ELBO: 2.4317 | CE: 0.4303 | KL: 2.0014


In [4]:

# ---- final evaluation ----
model.eval()
correct, total = 0, 0
all_probs, all_targets, causal_scores = [], [], []

with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(device), targets.to(device)
        logits, adj = model(images)
        probs = torch.softmax(logits, dim=-1)
        preds = probs.argmax(-1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        all_probs.append(probs); all_targets.append(targets)
        causal_scores.extend(verifier(model, images, logits, adj).cpu().tolist())

all_probs = torch.cat(all_probs); all_targets = torch.cat(all_targets)
ece, bins = expected_calibration_error(all_probs, all_targets)

sample_imgs = torch.stack([test_set[i][0] for i in range(min(32, len(test_set)))]).to(device)
unc = unc_engine.estimate(model, sample_imgs)

print("="*55)
print("AEGL FINAL EVALUATION REPORT")
print("="*55)
print(f"Test accuracy                 : {100*correct/total:.2f}%")
print(f"Expected Calibration Error     : {ece:.4f}")
print(f"Mean causal attribution (nats) : {sum(causal_scores)/len(causal_scores):.4f}")
print(f"Mean epistemic uncertainty     : {unc['epistemic_uncertainty'].mean().item():.4f}")
print(f"Mean aleatoric uncertainty     : {unc['aleatoric_uncertainty'].mean().item():.4f}")
print("="*55)


AEGL FINAL EVALUATION REPORT
Test accuracy                 : 71.71%
Expected Calibration Error     : 0.1616
Mean causal attribution (nats) : 0.1944
Mean epistemic uncertainty     : 0.0986
Mean aleatoric uncertainty     : 0.8660


## Where to go from here

- Swap `VisionEncoder` for a frozen CLIP/DINOv2 backbone to move from this
  8×8-digit demo to real open-world imagery (ImageNet-A/R, iNaturalist).
- The full interactive dashboard (`aegl/frontend/index.html` served by
  `aegl/backend/main.py`) exposes all of this — live training curves,
  reliability diagrams, and a per-sample evidence-graph inspector — without
  needing to re-run notebook cells.
